# AMD/NVIDIA 前瞻广度来源 Qlib 时钟修复

保留初版失败，取 SEC filing 后第一信号与独立受理审计时点的较晚者，来源事件定性缺口仍未知。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.nas_market_data import _settings,archive_nas_recorder
root=Path.cwd();card=root/'research/stage1_amd_nvda_guidance_breadth_qlib_map_v2_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
src=root/plan['source']['numeric_card'];clock=root/plan['timing']['audit_card'];orig=root/plan['source']['original_parent_card']
for path,key in [(src/'numeric_outlier_repair_result.json','final_numeric_result_sha256'),(src/'independent_numeric_source_review.json','independent_numeric_review_sha256'),(orig/'quarter_event_adjudication_result.json','original_quarter_event_result_sha256')]:assert sha(path)==plan['source'][key]
assert sha(clock/'result.json')==plan['timing']['audit_result_sha256']
numeric=json.loads((src/'numeric_outlier_repair_result.json').read_text());source=json.loads((orig/'quarter_event_adjudication_result.json').read_text());audit=json.loads((clock/'result.json').read_text());assert len(numeric['rows'])==len(source['selected_events'])==len(audit['rows'])==126 and numeric['counts']=={'AMD':{'numeric':61,'qualitative_unknown':2},'NVDA':{'numeric':63,'qualitative_unknown':0}}
assert source['parent_NAS_archive_sha256']==plan['source']['original_full_submission_NAS_archive_sha256'] and audit['status_counts']=={'conditional_date_gate_pass':126}
nby={x['accession']:x for x in numeric['rows']};cby={x['accession']:x for x in audit['rows']};assert len(nby)==len(cby)==126
state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['Qlib']['provider_uri']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(calendar)==plan['Qlib']['expected_full_rows']==3966 and calendar.is_unique and calendar.is_monotonic_increasing
rows=[]
for event in source['selected_events']:
 acc=event['accession'];n=nby[acc];c=cby[acc];assert event['issuer']==n['issuer']==c['issuer'] and event['filing_date']==n['filing_date'] and event['raw_sha256']==n['raw_sha256']==c['raw_sha256']
 filed=pd.Timestamp(event['filing_date']);later=calendar[calendar>filed];audit_day=pd.Timestamp(c['next_Qlib_session']);assert len(later)>0 and audit_day in calendar;available=max(later[0],audit_day);assert available>filed and available>=audit_day
 growth=n['growth'];assert (growth is None)==(n['status']!='numeric')
 rows.append({'issuer':event['issuer'],'accession':acc,'announcement_year':event['announcement_year'],'slot':event['slot'],'filing_date':filed,'first_eligible_signal_date':available,'valid_through_signal_date':filed+pd.DateOffset(months=4),'growth':float(growth) if growth is not None else np.nan,'source_numeric':growth is not None,'raw_sha256':event['raw_sha256']})
events=pd.DataFrame(rows).sort_values(['issuer','first_eligible_signal_date']).reset_index(drop=True);assert len(events)==126
assert events.groupby('issuer').size().to_dict()=={'AMD':63,'NVDA':63} and events.groupby('issuer').first_eligible_signal_date.apply(lambda x:x.is_unique).all()
left=pd.DataFrame({'signal_date':calendar});mapped=left.copy()
for issuer in ('AMD','NVDA'):
 z=events.loc[events.issuer.eq(issuer),['first_eligible_signal_date','filing_date','valid_through_signal_date','growth','accession']].sort_values('first_eligible_signal_date')
 aligned=pd.merge_asof(left,z,left_on='signal_date',right_on='first_eligible_signal_date',direction='backward',allow_exact_matches=True)
 known=aligned.growth.notna()&aligned.signal_date.le(aligned.valid_through_signal_date)&aligned.signal_date.ge(aligned.first_eligible_signal_date)
 mapped[f'{issuer}_source_known']=known.to_numpy(bool);mapped[f'{issuer}_growth']=aligned.growth.where(known).to_numpy(float);mapped[f'{issuer}_source_filing_date']=aligned.filing_date; mapped[f'{issuer}_source_accession']=aligned.accession
 assert mapped.loc[mapped[f'{issuer}_source_known'],'signal_date'].gt(mapped.loc[mapped[f'{issuer}_source_known'],f'{issuer}_source_filing_date']).all()
mapped['source_known']=mapped.AMD_source_known & mapped.NVDA_source_known
mapped['AMD_NVDA_forward_growth_positive_breadth']=((mapped.AMD_growth.gt(0).astype(float)+mapped.NVDA_growth.gt(0).astype(float))/2).where(mapped.source_known)
assert mapped.loc[mapped.source_known,'AMD_NVDA_forward_growth_positive_breadth'].isin((0.,0.5,1.)).all() and mapped.loc[~mapped.source_known,'AMD_NVDA_forward_growth_positive_breadth'].isna().all()
mapped.index=calendar;train=mapped.loc['2011-01-03':'2022-12-30'];held=mapped.loc['2023-01-03':'2026-10-07'];assert len(train)==3020 and len(held)==944
static=mapped[['AMD_NVDA_forward_growth_positive_breadth','source_known']].copy();static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument']);static=pd.concat({'feature':static[['AMD_NVDA_forward_growth_positive_breadth']],'meta':static[['source_known']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['Qlib']['train']),'held':tuple(plan['Qlib']['held_reused'])});assert len(dataset.prepare('train',col_set=['feature','meta']))==3020 and len(dataset.prepare('held',col_set=['feature','meta']))==944;dataset.config(dump_all=True,recursive=True)
report={'status':'original_AMD_NVDA_positive_guidance_breadth_Qlib_full_calendar_mapped_development_only','plan_sha256':sha(card/'plan.json'),'original_events':126,'events_delayed_beyond_audited_earliest_due_filing_date':int((events.first_eligible_signal_date>pd.to_datetime([cby[x].get('next_Qlib_session') for x in events.accession])).sum()),'numeric_events':124,'qualitative_unknown_events':2,'full_Qlib_calendar_rows':len(calendar),'train_calendar_rows':len(train),'held_calendar_rows':len(held),'known_train_sessions':int(train.source_known.sum()),'known_held_sessions':int(held.source_known.sum()),'unknown_train_sessions':int((~train.source_known).sum()),'unknown_held_sessions':int((~held.source_known).sum()),'train_breadth_counts':{str(v):int((train.AMD_NVDA_forward_growth_positive_breadth==v).sum()) for v in (0.,0.5,1.)},'held_breadth_counts':{str(v):int((held.AMD_NVDA_forward_growth_positive_breadth==v).sum()) for v in (0.,0.5,1.)},'first_known_signal_date':str(mapped.loc[mapped.source_known,'signal_date'].iloc[0].date()),'issuer_first_public_exact_clock_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'source_audit.json').write_text(json.dumps(report,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes=plan['source']);rec.save_objects(artifact_path='sources',events=events);rec.save_objects(artifact_path='analysis',mapped_signals=mapped,source_audit=report);rec.save_objects(artifact_path='metadata',dataset=dataset);rec.log_metrics(source_events=126,known_train_days=report['known_train_sessions'],known_held_days=report['known_held_sessions'],model_fits=0,native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid);assert len(rec.load_object('sources/events'))==126 and len(rec.load_object('analysis/mapped_signals'))==3966 and len(rec.load_object('metadata/dataset').prepare('held',col_set=['feature','meta']))==944
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='amd-nvda-guidance-map-restore-') as td:
 archive=Path(td)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:names=[m.name for m in z.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/sources/events') for x in names) and any(x.endswith('/artifacts/analysis/mapped_signals') for x in names) and any(x.endswith('/artifacts/metadata/dataset') for x in names)
summary={'experiment':plan['experiment'],'decision':report['status'],'objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state.update(status='completed',recorder_id=rid,nas_archive_sha256=evidence['sha256']);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(report);print({'recorder_id':rid,'NAS':evidence['sha256']})


{'status': 'original_AMD_NVDA_positive_guidance_breadth_Qlib_full_calendar_mapped_development_only', 'plan_sha256': '2a2496a90526bd35db077a5a5a206622c8b5759037c30591569c2e8dd3594b0a', 'original_events': 126, 'events_delayed_beyond_audited_earliest_due_filing_date': 1, 'numeric_events': 124, 'qualitative_unknown_events': 2, 'full_Qlib_calendar_rows': 3966, 'train_calendar_rows': 3020, 'held_calendar_rows': 944, 'known_train_sessions': 2881, 'known_held_sessions': 944, 'unknown_train_sessions': 139, 'unknown_held_sessions': 0, 'train_breadth_counts': {'0.0': 772, '0.5': 951, '1.0': 1158}, 'held_breadth_counts': {'0.0': 0, '0.5': 396, '1.0': 548}, 'first_known_signal_date': '2011-07-22', 'issuer_first_public_exact_clock_qualified': False, 'model_fits': 0, 'native_accounts': 0, 'market_outcome_reads': 0}
{'recorder_id': '78a3e941161c4ec598fe29894e5af76d', 'NAS': 'ac73a3a5de924f85ac589f8bfba43d41db5d34a17c0571577088f73f331d2510'}


## 中文结论

保守时钟 v2 Qlib 来源映射通过：训练 2881/3020、复用评价 944/944 日已知；一条 AMD 定性事件按 filing date 比审计下界再晚一日，原版失败保留。Recorder/NAS 回读通过；零拟合、零账户，详见 `conclusion.md`。